# Gobierno · Comentarios, tags y llaves en Gold

## Propósito

Documentar y gobernar las tablas de la capa Gold desde Unity Catalog: cada tabla lleva su comentario con el grano y el owner, su clasificación y sus llaves primarias y foráneas informativas.

## Flujo

1. Aplicar a cada tabla Gold su **COMMENT** y los tags `clasificacion` y `layer`.
2. Declarar la **llave primaria** de cada tabla que la tiene, solo si la llave es única y no tiene nulos.
3. Declarar las **llaves foráneas informativas** entre hechos y dimensiones.

## Destino

Catálogo `gld_dataco`, esquema `supply_chain`. No modifica datos, solo metadatos.

## Reglas

* Las llaves son informativas: Unity Catalog no las exige al escribir, pero documentan el modelo y alimentan el linaje.
* No hay restricciones `CHECK` en las tablas con máscara de columna, porque Databricks no permite combinarlas.
* El notebook es idempotente: si una restricción ya existe, lo informa y continúa.

## Resultado esperado

Una línea por tabla y acción con el estado `OK`, `OMITIDO` o `YA EXISTE/ERROR`.

In [0]:
G = "gld_dataco.supply_chain"

TABLAS = {
  # tabla: (comentario, clasificación, llave primaria)
  "desc_dim_customers":    ("Dimensión de clientes con historial SCD2. Grano: una fila por versión de cliente. Contiene datos personales. Owner: ingeniería de datos.", "confidencial", "cliente_key"),
  "desc_dim_products":     ("Dimensión de productos con historial SCD2. Grano: una fila por versión de producto. Owner: ingeniería de datos.", "interno", "producto_key"),
  "desc_dim_locations":    ("Dimensión de ubicaciones. Grano: una fila por combinación de market, región, país, estado, ciudad y código postal. Owner: ingeniería de datos.", "interno", "location_key"),
  "desc_dim_fecha":        ("Dimensión de fechas. Grano: una fila por fecha. Owner: ingeniería de datos.", "interno", "fecha_key"),
  "desc_dim_shipping_mode":("Dimensión de modos de envío. Grano: una fila por modo de envío. Owner: ingeniería de datos.", "interno", "shipping_mode_key"),
  "desc_ft_orders":        ("Hechos de pedidos con historial SCD2. Grano: una fila por versión de pedido. Owner: ingeniería de datos.", "interno", "orden_key"),
  "desc_ft_order_details": ("Hechos de líneas de pedido. Grano: una fila por línea de pedido. Owner: ingeniería de datos.", "interno", None),
  "desc_ft_order_summary": ("Resumen por pedido, base única de los indicadores. Owner: ingeniería de datos.", "interno", None),
  "desc_ft_kpi_resumen_ejecutivo": ("Indicadores del resumen ejecutivo, con historial por corte. Owner: ingeniería de datos.", "interno", None),
  "desc_ft_kpi_detalle_region":    ("Indicadores por región, con historial por corte. Owner: ingeniería de datos.", "interno", None),
  "desc_ft_kpi_detalle_orden":     ("Indicadores por orden con historial SCD2. Owner: ingeniería de datos.", "interno", "orden_detalle_key"),
  "desc_ft_kpi_alertas":           ("Alertas de indicadores (snapshot por corte). Owner: ingeniería de datos.", "interno", None),
}

FKS = [
  ("desc_ft_orders", "cliente_key", "desc_dim_customers", "cliente_key"),
  ("desc_ft_orders", "location_key", "desc_dim_locations", "location_key"),
  ("desc_ft_orders", "shipping_mode_key", "desc_dim_shipping_mode", "shipping_mode_key"),
  ("desc_ft_orders", "fecha_pedido_key", "desc_dim_fecha", "fecha_key"),
  ("desc_ft_orders", "fecha_envio_key", "desc_dim_fecha", "fecha_key"),
  ("desc_ft_order_details", "orden_key", "desc_ft_orders", "orden_key"),
  ("desc_ft_order_details", "producto_key", "desc_dim_products", "producto_key"),
]

resultados = []
def registrar(t, accion, estado):
    resultados.append((t, accion, estado)); print(f"{estado:<9} {t} · {accion}")

# 1) Comentario y tags
for t, (com, clasif, pk) in TABLAS.items():
    try:
        spark.sql(f"COMMENT ON TABLE {G}.{t} IS '{com}'")
        spark.sql(f"ALTER TABLE {G}.{t} SET TAGS ('clasificacion'='{clasif}', 'layer'='gold')")
        registrar(t, "comentario y tags", "OK")
    except Exception as e:
        registrar(t, "comentario y tags", "ERROR: " + str(e)[:120])

# 2) Llaves primarias (solo si la llave es única y sin nulos)
for t, (com, clasif, pk) in TABLAS.items():
    if not pk: continue
    try:
        if pk not in spark.table(f"{G}.{t}").columns:
            registrar(t, f"PK {pk}", "OMITIDO (columna no existe)"); continue
        n, d, nulos = spark.sql(f"SELECT COUNT(*), COUNT(DISTINCT {pk}), SUM(CASE WHEN {pk} IS NULL THEN 1 ELSE 0 END) FROM {G}.{t}").first()
        if d != n or nulos:
            registrar(t, f"PK {pk}", f"OMITIDO (no única o con nulos: {n} filas, {d} distintas)"); continue
        spark.sql(f"ALTER TABLE {G}.{t} ALTER COLUMN {pk} SET NOT NULL")
        spark.sql(f"ALTER TABLE {G}.{t} ADD CONSTRAINT pk_{t} PRIMARY KEY ({pk})")
        registrar(t, f"PK {pk}", "OK")
    except Exception as e:
        registrar(t, f"PK {pk}", "YA EXISTE/ERROR: " + str(e)[:100])

# 3) Llaves foráneas informativas
for t, col, ref, refcol in FKS:
    try:
        if col not in spark.table(f"{G}.{t}").columns:
            registrar(t, f"FK {col}", "OMITIDO (columna no existe)"); continue
        spark.sql(f"ALTER TABLE {G}.{t} ADD CONSTRAINT fk_{t}_{col} FOREIGN KEY ({col}) REFERENCES {G}.{ref}({refcol})")
        registrar(t, f"FK {col} -> {ref}", "OK")
    except Exception as e:
        registrar(t, f"FK {col} -> {ref}", "YA EXISTE/ERROR: " + str(e)[:100])

# Gobierno · Máscara de datos personales y permisos

## Propósito

Proteger el correo del cliente y dar acceso de lectura al grupo consumidor de Gold.

## Flujo

1. Crear la función `mask_email`: los miembros de `dac_lectores` ven `***@***`; los demás ven el correo real.
2. Marcar la columna `customer_email` con el tag `DAC = YES` y aplicarle la máscara.
3. Dar al grupo `dac_lectores` permiso de lectura (`USE CATALOG`, `USE SCHEMA` y `SELECT`) sobre Gold.

## Destino

`gld_dataco.supply_chain.desc_dim_customers`, columna `customer_email`.

## Control

Después de aplicar la máscara, se ejecuta `gld_dim_customers` para comprobar que el pipeline puede seguir escribiendo en la tabla. Para revertir la máscara: `ALTER TABLE gld_dataco.supply_chain.desc_dim_customers ALTER COLUMN customer_email DROP MASK`.

## Resultado esperado

Los mensajes `OK máscara y tag DAC en customer_email` y `OK permisos para dac_lectores`.

In [0]:
G = "gld_dataco.supply_chain"

# Máscara del correo: los miembros de dac_lectores ven el correo oculto
spark.sql(f"""
CREATE OR REPLACE FUNCTION {G}.mask_email(email STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('dac_lectores') THEN '***@***' ELSE email END
""")

cols = spark.table(f"{G}.desc_dim_customers").columns
assert "customer_email" in cols, "No existe customer_email en desc_dim_customers"

spark.sql(f"ALTER TABLE {G}.desc_dim_customers ALTER COLUMN customer_email SET TAGS ('DAC' = 'YES')")
spark.sql(f"ALTER TABLE {G}.desc_dim_customers ALTER COLUMN customer_email SET MASK {G}.mask_email")
print("OK máscara y tag DAC en customer_email")

# Permisos: el grupo lector consulta Gold
spark.sql("GRANT USE CATALOG ON CATALOG gld_dataco TO `dac_lectores`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {G} TO `dac_lectores`")
spark.sql(f"GRANT SELECT ON SCHEMA {G} TO `dac_lectores`")
print("OK permisos para dac_lectores")